[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/23_probabilistic_machine_learning/05_hidden_markov_models.ipynb)

# Bài 05 — Hidden Markov Model

**🎯 Mục tiêu**
- Chuỗi Markov: ma trận chuyển, phân phối dừng.
- HMM: trạng thái ẩn sinh ra quan sát; tham số $(\boldsymbol\pi, A, B)$.
- Ba bài toán kinh điển: **đánh giá** (forward), **giải mã** (Viterbi, forward–backward), **học** (Baum–Welch = EM) — cài đặt from scratch, kiểm chứng bằng vét cạn.
- Ứng dụng, giới hạn và vị trí của HMM so với mô hình chuỗi hiện đại.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/23_probabilistic_machine_learning

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import logsumexp

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Chuỗi Markov

Trạng thái thời tiết hôm nay chỉ phụ thuộc hôm qua (**tính Markov**): $P(z_t \mid z_{t-1}, z_{t-2}, \dots) = P(z_t \mid z_{t-1}) = A_{z_{t-1}, z_t}$.

In [ ]:
states = ["nắng", "mây", "mưa"]
A_w = np.array([[0.7, 0.2, 0.1],
                [0.3, 0.4, 0.3],
                [0.2, 0.3, 0.5]])                                      # hàng: hôm qua, cột: hôm nay
z = [0]
for _ in range(20000):
    z.append(rng.choice(3, p=A_w[z[-1]]))
empirical = np.bincount(z, minlength=3) / len(z)
vals, vecs = np.linalg.eig(A_w.T)                                      # phân phối dừng: π A = π
stationary = np.real(vecs[:, np.argmin(np.abs(vals - 1))])
stationary /= stationary.sum()
print("phân phối dừng (vector riêng trái, trị riêng 1):", dict(zip(states, stationary.round(3))))
print("tần suất mô phỏng 20 000 ngày:              ", dict(zip(states, empirical.round(3))))
print("sau 2 ngày từ 'mưa': P =", dict(zip(states, np.linalg.matrix_power(A_w, 2)[2].round(3))))

Phân phối $n$ bước sau: $\mathbf p_{t+n} = \mathbf p_t A^n$; với chuỗi "tốt" (tối giản, không tuần hoàn), $\mathbf p_t$ hội tụ về **phân phối dừng** bất kể điểm xuất phát — nền tảng của PageRank và MCMC.

## 2. Hidden Markov Model: sòng bạc gian lận

Sòng bạc dùng xen kẽ hai con xúc xắc: **công bằng** (mỗi mặt 1/6) và **gian lận** (mặt 6 xác suất 1/2). Ta chỉ thấy kết quả tung; con xúc xắc đang dùng là **ẩn** (Durbin et al., 1998).

| Thành phần | Ký hiệu | Ở đây |
|---|---|---|
| Phân phối ban đầu | $\pi_i = P(z_1 = i)$ | [0.5, 0.5] |
| Ma trận chuyển | $A_{ij} = P(z_t = j \mid z_{t-1} = i)$ | đổi xúc xắc: công bằng→gian lận 0.05, gian lận→công bằng 0.1 |
| Phát xạ | $B_{ik} = P(x_t = k \mid z_t = i)$ | công bằng: 1/6 mỗi mặt; gian lận: mặt 6 = 0.5, còn lại 0.1 |

In [ ]:
pi_true = np.array([0.5, 0.5])
A_true = np.array([[0.95, 0.05], [0.10, 0.90]])
B_true = np.array([[1 / 6] * 6, [0.1] * 5 + [0.5]])


def sample_hmm(pi, A, B, T, r):
    z = np.empty(T, int)
    x = np.empty(T, int)
    z[0] = r.choice(len(pi), p=pi)
    for t in range(T):
        if t:
            z[t] = r.choice(len(pi), p=A[z[t - 1]])
        x[t] = r.choice(B.shape[1], p=B[z[t]])
    return z, x


z_true, rolls = sample_hmm(pi_true, A_true, B_true, 400, rng)
print("80 lần tung đầu (mặt 1–6):", "".join(str(v + 1) for v in rolls[:80]))
print("xúc xắc thật (0 = công bằng, 1 = gian lận):", "".join(str(v) for v in z_true[:80]))
print(f"tỉ lệ thời gian dùng xúc xắc gian lận: {z_true.mean():.2f}")

## 3. Đánh giá: thuật toán forward

$P(\mathbf x_{1:T}) = \sum_{\mathbf z} P(\mathbf x, \mathbf z)$ — vét cạn có $K^T$ chuỗi trạng thái ($2^{400}$!). **Forward** dùng quy hoạch động: $\alpha_t(j) = P(x_{1:t}, z_t = j)$
$$\alpha_1(j) = \pi_j B_{j,x_1}, \qquad \alpha_t(j) = \Big[\sum_i \alpha_{t-1}(i)A_{ij}\Big]B_{j,x_t}, \qquad P(\mathbf x) = \sum_j\alpha_T(j)$$
Độ phức tạp $O(TK^2)$. Tính trong **không gian log** để tránh underflow (xác suất của 400 lần tung ~ $10^{-300}$): $\log\sum_i e^{a_i}A_{ij} = c + \log\sum_i e^{a_i - c}A_{ij}$ với $c = \max_i a_i$ (mẹo log-sum-exp) — vẫn là một phép nhân ma trận–vector, nhanh.

In [ ]:
def forward_log(x, pi, A, B):
    T, K = len(x), len(pi)
    log_B = np.log(B)
    log_alpha = np.empty((T, K))
    log_alpha[0] = np.log(pi) + log_B[:, x[0]]
    for t in range(1, T):
        c = log_alpha[t - 1].max()                                     # log-sum-exp: trừ max để exp không underflow
        log_alpha[t] = c + np.log(np.exp(log_alpha[t - 1] - c) @ A) + log_B[:, x[t]]
    return log_alpha, logsumexp(log_alpha[-1])


def brute_force_log_prob(x, pi, A, B):
    total = []
    for zs in itertools.product(range(len(pi)), repeat=len(x)):
        lp = np.log(pi[zs[0]]) + np.log(B[zs[0], x[0]])
        for t in range(1, len(x)):
            lp += np.log(A[zs[t - 1], zs[t]]) + np.log(B[zs[t], x[t]])
        total.append(lp)
    return logsumexp(total)


short = rolls[:10]
print(f"10 lần tung: forward = {forward_log(short, pi_true, A_true, B_true)[1]:.6f} | vét cạn 2¹⁰ = 1024 chuỗi = {brute_force_log_prob(short, pi_true, A_true, B_true):.6f}")
_, ll_true = forward_log(rolls, pi_true, A_true, B_true)
_, ll_fair = forward_log(rolls, np.array([1.0]), np.array([[1.0]]), B_true[:1])     # mô hình 1 trạng thái: luôn công bằng
print(f"400 lần tung: log P(x | HMM thật) = {ll_true:.2f} | log P(x | luôn công bằng) = {ll_fair:.2f}")

Log-likelihood dùng để **so sánh mô hình** ("chuỗi này giống sòng bạc gian lận hay công bằng?") — ví dụ phân loại chuỗi: mỗi lớp một HMM, chọn lớp có likelihood cao nhất (nhận dạng tiếng nói cổ điển).

## 4. Giải mã: Viterbi và forward–backward

Hai câu hỏi khác nhau:
- **Viterbi**: chuỗi trạng thái **đồng thời** khả dĩ nhất $\arg\max_{\mathbf z}P(\mathbf z \mid \mathbf x)$ — thay tổng bằng max trong forward, lưu con trỏ để truy ngược.
- **Posterior decoding** (forward–backward): với **từng** $t$, $P(z_t = j \mid \mathbf x) \propto \alpha_t(j)\beta_t(j)$ với $\beta_t(i) = P(x_{t+1:T} \mid z_t = i)$.

In [ ]:
def viterbi(x, pi, A, B):
    T, K = len(x), len(pi)
    delta = np.empty((T, K))
    back = np.zeros((T, K), int)
    delta[0] = np.log(pi) + np.log(B[:, x[0]])
    for t in range(1, T):
        scores = delta[t - 1][:, None] + np.log(A)                     # [i, j]: đến j từ i
        back[t] = scores.argmax(0)
        delta[t] = scores.max(0) + np.log(B[:, x[t]])
    path = np.empty(T, int)
    path[-1] = delta[-1].argmax()
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def backward_log(x, A, B):
    T, K = len(x), A.shape[0]
    log_B = np.log(B)
    log_beta = np.zeros((T, K))
    for t in range(T - 2, -1, -1):
        b = log_B[:, x[t + 1]] + log_beta[t + 1]
        c = b.max()
        log_beta[t] = c + np.log(A @ np.exp(b - c))
    return log_beta


def posterior_states(x, pi, A, B):
    log_alpha, ll = forward_log(x, pi, A, B)
    return np.exp(log_alpha + backward_log(x, A, B) - ll)


path = viterbi(rolls, pi_true, A_true, B_true)
gamma = posterior_states(rolls, pi_true, A_true, B_true)
print(f"Viterbi đúng {np.mean(path == z_true):.1%} số lần tung | posterior decoding đúng {np.mean(gamma.argmax(1) == z_true):.1%}")
print(f"đoán 'gian lận' nếu mặt 6 (không dùng HMM): đúng {np.mean((rolls == 5) == z_true):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 3.5))
ax.fill_between(np.arange(400), 0, z_true, step="mid", color="gray", alpha=0.3, label="đang dùng xúc xắc gian lận (thật)")
ax.plot(gamma[:, 1], color="r", label="P(gian lận | toàn bộ chuỗi) — forward–backward")
ax.plot(path * 1.05, color="b", lw=1, label="Viterbi")
ax.set_xlabel("lần tung")
ax.legend(loc="upper right", fontsize=8)
plt.show()

HMM tận dụng **tính liên tục** của trạng thái: một mặt 6 đơn lẻ không đủ kết luận, nhưng một chuỗi nhiều mặt 6 thì có. Posterior decoding cho thêm **độ chắc chắn** từng thời điểm (gần ranh giới chuyển trạng thái, xác suất lưng chừng).

## 5. Học: Baum–Welch (EM cho HMM)

Không biết $(\boldsymbol\pi, A, B)$, chỉ có chuỗi quan sát. EM (bài 02) với biến ẩn là chuỗi trạng thái:
- **E-step**: forward–backward → $\gamma_t(i) = P(z_t = i \mid \mathbf x)$ và $\xi_t(i, j) = P(z_t = i, z_{t+1} = j \mid \mathbf x)$.
- **M-step**: $A_{ij} \propto \sum_t\xi_t(i, j)$, $B_{ik} \propto \sum_{t: x_t = k}\gamma_t(i)$, $\pi_i = \gamma_1(i)$.

In [ ]:
def baum_welch(x, K, M, n_iter=200, seed=0, tol=1e-6):
    r = np.random.default_rng(seed)
    pi = np.full(K, 1 / K)
    A = r.dirichlet(np.ones(K) * 5, K)
    B = r.dirichlet(np.ones(M) * 5, K)
    history = []
    for _ in range(n_iter):
        log_alpha, ll = forward_log(x, pi, A, B)
        log_beta = backward_log(x, A, B)
        gamma = np.exp(log_alpha + log_beta - ll)
        log_xi = (log_alpha[:-1, :, None] + np.log(A)[None] + np.log(B[:, x[1:]]).T[:, None, :] + log_beta[1:, None, :] - ll)
        xi = np.exp(log_xi)
        history.append(ll)
        pi = gamma[0]
        A = xi.sum(0) / xi.sum(0).sum(1, keepdims=True)
        B = np.array([[gamma[x == k, i].sum() for k in range(M)] for i in range(K)])
        B /= B.sum(1, keepdims=True)
        if len(history) > 1 and abs(history[-1] - history[-2]) < tol:
            break
    return pi, A, B, np.array(history)


_, long_rolls = sample_hmm(pi_true, A_true, B_true, 5000, np.random.default_rng(1))
pi_hat, A_hat, B_hat, hist = baum_welch(long_rolls, K=2, M=6, seed=0)
loaded = int(np.argmax(B_hat[:, 5]))                                   # trạng thái nào là "gian lận"? (thứ tự trạng thái tuỳ ý)
order = [1 - loaded, loaded]
print(f"{len(hist)} vòng EM | log-likelihood tăng đơn điệu: {bool(np.all(np.diff(hist) > -1e-6))}")
print("A học được (công bằng, gian lận):\n", A_hat[np.ix_(order, order)].round(3), "\nA thật:\n", A_true)
print("B học được:\n", B_hat[order].round(3))

Từ 5 000 lần tung không nhãn, Baum–Welch khôi phục khá sát cả ma trận chuyển lẫn xúc xắc gian lận. Lưu ý:
- **Hoán vị nhãn** (label switching): EM không biết trạng thái nào tên là "gian lận" — ta tự gán nghĩa sau khi học (ở đây: trạng thái có $B_{\cdot,6}$ lớn).
- **Cực trị địa phương** → nhiều khởi tạo; cần **nhiều dữ liệu** (chuyển trạng thái hiếm → ít thông tin về $A$).
- Phát xạ có thể là Gaussian (giá cổ phiếu, cảm biến) hoặc GMM — thư viện `hmmlearn` cài sẵn các biến thể.

## 6. Ứng dụng và vị trí hiện nay

| Lĩnh vực | Trạng thái ẩn | Quan sát |
|---|---|---|
| Nhận dạng tiếng nói (cổ điển) | âm vị | đặc trưng âm thanh |
| Gán nhãn từ loại (POS) | từ loại | từ |
| Tài chính | chế độ thị trường (tăng/giảm/biến động) | lợi suất |
| Sinh học | vùng gen mã hoá / không mã hoá | nucleotide |
| Giám sát thiết bị | trạng thái máy (tốt/xuống cấp) | cảm biến |

**Giới hạn**: Markov bậc 1 (nhớ ngắn), quan sát độc lập có điều kiện theo trạng thái, số trạng thái rời rạc nhỏ. Trong NLP và tiếng nói, HMM đã được thay bằng CRF, RNN/LSTM (chương 30) và Transformer (chương 31). HMM vẫn hữu ích khi cần mô hình **nhỏ, diễn giải được**, ít dữ liệu, hoặc phát hiện **chế độ** (regime). Kalman filter là "HMM liên tục" (trạng thái Gaussian tuyến tính) — dùng chính công thức Gaussian có điều kiện của bài 01.

## 🧠 Kiểm tra nhanh

1. Tính Markov là gì? Phân phối dừng tính thế nào?
2. Ba bài toán của HMM và thuật toán tương ứng?
3. Vì sao forward là $O(TK^2)$ thay vì $O(K^T)$? Vì sao tính trong log?
4. Viterbi khác posterior decoding thế nào?
5. Baum–Welch liên hệ với EM ra sao? "Label switching" là gì?

## 📌 Tóm tắt

- HMM = chuỗi Markov ẩn + phát xạ; tham số $(\boldsymbol\pi, A, B)$.
- Forward (đánh giá), Viterbi / forward–backward (giải mã), Baum–Welch (học = EM) — đều là quy hoạch động $O(TK^2)$.
- Nhỏ, diễn giải được, hợp phát hiện chế độ; NLP/tiếng nói hiện đại dùng mô hình nơ-ron.

## 📚 Tài liệu tham khảo

- Rabiner (1989) — *A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition*
- Durbin, Eddy, Krogh & Mitchison (1998) — *Biological Sequence Analysis*, ch. 3 (sòng bạc gian lận)
- Jurafsky & Martin — *Speech and Language Processing*, Appendix A (HMM)